# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShayanAhmed12/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** CTR / Engagement Opportunity Scoring (provisional, from ML-02).

**Task type: ranking / scoring**, built on two pieces:

1. **Expected-CTR model (regression):** for each visible page, estimate what CTR a page *like it* would normally get, given its position, impressions, content type, intent, age. The gap between actual and expected CTR is the page's **opportunity**.
2. **Review queue (ranking):** sort pages by *clicks at stake* = (expected − actual CTR) × impressions, and evaluate only the top of the list, because an editor only ever reviews the top K.

Why ranking and not plain classification: the editor's question is *"which pages first?"*, not *"is this page good or bad?"*. Capacity is limited (say 50 pages a week), so what matters is how good the top of the list is. Why not clustering: clusters would describe page types, but wouldn't order them for action.


In [1]:
# Setup: load the starter dataset (runs locally or in Colab)
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The task in one table
task = pd.DataFrame({
    "piece": ["task type", "unit", "model output", "queue score", "evaluated on"],
    "choice": ["ranking / scoring (regression for expected CTR + ranked queue)",
               "one page, with a 30-day feature window and the next 30-day outcome window",
               "expected CTR for a page like this",
               "clicks at stake = (expected - actual CTR) x impressions",
               "top-K of the queue, on clients the model never saw"],
})
print(task.to_string(index=False))


       piece                                                                    choice
   task type            ranking / scoring (regression for expected CTR + ranked queue)
        unit one page, with a 30-day feature window and the next 30-day outcome window
model output                                         expected CTR for a page like this
 queue score                   clicks at stake = (expected - actual CTR) x impressions
evaluated on                        top-K of the queue, on clients the model never saw


## 2. Target or proxy

The trap here: if I define "under-performing" with a rule (e.g. `ctr < 0.5`) using the **same** 90-day numbers I feed in as features, the model just learns my rule back. That would be circular.

**What I use instead: a label measured in a later window.** The starter data splits recent activity into two windows: `*_prev_30d` (days 31–60 ago) and `*_last_30d` (the last 30 days). So:

- **Features window:** `impressions_prev_30d`, `clicks_prev_30d` (→ CTR in the earlier month), plus slow-moving context (content type, intent, age, word count, average position).
- **Target (observed outcome):** `under_captures_next` = 1 if, in the **last 30 days**, the page's CTR was below half the median CTR of comparable pages (same position tier), with at least 100 impressions in that window.

This is still a **proxy**. "Under-captures clicks next month" is not the same as "would gain clicks if an editor rewrote the title". It's the closest outcome the data actually observes. It's also only *partly* clean: `avg_position` / `position_tier` are 90-day averages that overlap the outcome window. I keep them because a page's position barely moves inside 90 days, but I'll flag it as a leakage risk for the warehouse version, where I can compute position from the feature window only.


In [2]:
# Build the forward-looking label
d = df.copy()
d["ctr_prev"] = np.where(d["impressions_prev_30d"] > 0, 100 * d["clicks_prev_30d"] / d["impressions_prev_30d"], np.nan)
d["ctr_last"] = np.where(d["impressions_last_30d"] > 0, 100 * d["clicks_last_30d"] / d["impressions_last_30d"], np.nan)

# Lane slice: visible pages with enough volume in BOTH windows, real position data, position 1-20
lane = d[(d["impressions_prev_30d"] >= 100) & (d["impressions_last_30d"] >= 100)
         & (d["avg_position"] > 0) & (d["avg_position"] <= 20)].copy()

# Expected CTR per tier, computed separately inside each window
lane["exp_prev"] = lane.groupby("position_tier")["ctr_prev"].transform("median")
lane["exp_last"] = lane.groupby("position_tier")["ctr_last"].transform("median")
lane["under_captures_next"] = (lane["ctr_last"] < 0.5 * lane["exp_last"]).astype(int)

print(f"Lane slice: {len(lane):,} pages from {lane['client_id'].nunique()} clients")
print(f"Base rate of the target (under-captures next month): {lane['under_captures_next'].mean():.3f}")
print("\nTarget rate by position tier:")
print(lane.groupby("position_tier")["under_captures_next"].agg(["size", "mean"]).round(3).to_string())


Lane slice: 11,111 pages from 29 clients
Base rate of the target (under-captures next month): 0.347

Target rate by position tier:
               size   mean
position_tier             
page_1         6575  0.312
page_3_5         15  0.467
striking       4168  0.405
top_3           353  0.292


## 3. Success metric

**Primary metric: Precision@50 of the review queue, on held-out clients.** Of the 50 pages the queue tells an editor to review first, how many really did under-capture clicks in the outcome month?

- Why @50: that's roughly one editor's weekly review capacity, and only the top of the list gets acted on.
- Why held-out clients: in practice the queue is used on new sites. Testing on clients it trained on would flatter it (notebook 02 showed the gap shrinks on unseen clients).
- **What "good" means:** the queue must clearly beat (a) the **base rate** (random pages) and (b) the **simple rule** an editor might already use: "lowest CTR in the earlier month first, among pages with ≥500 impressions". I'll call it worth building only if it beats the rule by about 10 points of Precision@50 averaged over several client splits, not on one lucky split.
- **Secondary:** *clicks at stake captured in the top 50*, which checks that the queue picks pages where fixing things would matter, not only small pages with technically low CTR.

The cell below computes the two reference numbers the model will have to beat.


In [3]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

y = lane["under_captures_next"].values
# Reference 1: random pages = base rate
print(f"Base rate (random queue):              {y.mean():.3f}")
# Reference 2: a simple rule - lowest earlier-month CTR first, among pages with >=500 earlier impressions
rule_score = np.where(lane["impressions_prev_30d"] >= 500, -lane["ctr_prev"], -np.inf)
print(f"Simple rule Precision@50 (all pages):   {precision_at_k(rule_score, y, 50):.3f}")
# Reference 3: the 'persistence' version - earlier-month gap vs tier expectation, weighted by impressions
gap_score = (lane["exp_prev"] - lane["ctr_prev"]) / 100 * lane["impressions_prev_30d"]
print(f"Tier-gap x impressions Precision@50:   {precision_at_k(gap_score, y, 50):.3f}")
print("\n-> the model has to beat these, on held-out clients, to earn its place.")


Base rate (random queue):              0.347
Simple rule Precision@50 (all pages):   0.500
Tier-gap x impressions Precision@50:   0.840

-> the model has to beat these, on held-out clients, to earn its place.


**What these reference numbers say (in-sample, all clients, one 30-day hop):** random pages hit 0.347. The plain "lowest CTR first" rule reaches 0.50. The position-adjusted gap × impressions score already reaches **0.84**: earlier-month under-capture on big pages is very persistent. So the bar a model has to clear is the tier-gap baseline, not the naive rule. That's a high bar. A realistic win for ML here is extra signal from content type, intent and volume (catching under-performers the tier median misses), measured on held-out clients. If it can't beat 0.84 there, the honest result is "the transparent baseline is the product". These numbers are in-sample on the starter slice and must be re-measured with client-holdout and on the warehouse.

## 4. The unit of analysis, as a real dataframe

**One row = one content page** (pseudonymized `content_id`, grouped under a pseudonymized `client_id`), with:
- its **feature window** (the earlier 30 days plus slow context),
- its **outcome window** (the last 30 days → the target),
- the **queue score** an editor would sort by.

`client_id` is used only to split train/test by client, never as a feature.


In [4]:
cols = ["content_id", "client_id", "content_type", "main_intent", "position_tier", "avg_position",
        "impressions_prev_30d", "clicks_prev_30d", "ctr_prev", "exp_prev",
        "impressions_last_30d", "ctr_last", "exp_last", "under_captures_next"]
unit = lane[cols].copy()
unit["clicks_at_stake_prev"] = ((unit["exp_prev"] - unit["ctr_prev"]).clip(lower=0) / 100 * unit["impressions_prev_30d"]).round(1)
assert unit["content_id"].is_unique, "one row must be exactly one page"
print(f"Shape: {unit.shape}  |  unique pages: {unit['content_id'].nunique():,}  |  one row = one page")
unit.sort_values("clicks_at_stake_prev", ascending=False).head(8).round(3)


Shape: (11111, 15)  |  unique pages: 11,111  |  one row = one page


,content_id,client_id,content_type,main_intent,position_tier,avg_position,impressions_prev_30d,clicks_prev_30d,ctr_prev,exp_prev,impressions_last_30d,ctr_last,exp_last,under_captures_next,clicks_at_stake_prev
6653,content_5fe46e04994d,client_4e07408562,keyword article,informational,page_1,4.2,218786,250,0.114,0.234,120791,0.182,0.276,0,261.9
7445,content_c8e9d6ab9013,client_19581e27de,keyword article,informational,page_1,9.7,111885,0,0.000,0.234,63326,0.000,0.276,1,261.8
7678,content_8451fc6f034d,client_d029fa3a95,keyword article,informational,top_3,2.3,97606,30,0.031,0.254,168958,0.021,0.515,1,218.4
3394,content_36ff89c8214e,client_19581e27de,keyword article,informational,page_1,7.3,106412,63,0.059,0.234,106985,0.033,0.276,1,186.0
6903,content_c84a0ab98e90,client_f369cb89fc,keyword article,informational,page_1,7.8,84773,19,0.022,0.234,99337,0.036,0.276,1,179.4
482,content_39881853ef0c,client_f369cb89fc,keyword article,informational,page_1,7.2,64917,6,0.009,0.234,37677,0.003,0.276,1,145.9
3070,content_91652435f57a,client_19581e27de,keyword article,commercial,page_1,7.8,75160,31,0.041,0.234,74135,0.066,0.276,1,144.9
22028,content_73c54f78c06a,client_f369cb89fc,keyword article,informational,page_1,4.7,97200,84,0.086,0.234,85885,0.098,0.276,1,143.4


## 5. Why ML beats a fixed rule here

A fixed rule like `ctr < 0.5 and impressions >= 500` breaks for several reasons at once:

- **Position:** a 0.3% CTR is fine at position 15 and bad at position 2. The rule needs one threshold per tier.
- **Volume:** with 120 impressions, one extra click moves CTR by ~0.8 points, so small pages flip in and out of "low CTR" by chance. The rule needs a confidence adjustment.
- **Content type and intent:** comparison articles, feedly articles and keyword articles have different "normal" CTRs (notebook 01). Informational and transactional intent differ too. That's another split.
- **Persistence:** what matters is whether a gap *stays*. Some low-CTR months are one-off dips that fix themselves.

Each of these alone is an `if`. Together, the "normal CTR" depends on combinations of position × volume × type × intent × age, which is too many tangled cells to hand-tune. That's exactly what a model is for: learn the expected CTR from comparable pages, then rank by the gap.

**But ML only wins if it measurably beats the rule.** The plan is: tier-median baseline first, then a simple model (e.g. gradient boosting on expected CTR), both judged by Precision@50 on held-out clients. If the baseline is as good, I'll ship the baseline and say so. The code below backs this. In the earlier month, under-capturing pages with ≥500 impressions still under-captured the next month 57% of the time, against a 35% base rate: the signal is real but far from certain. Medians also differ by content type within the same tier (e.g. page_1: keyword 0.235% vs feedly 0.299% vs comparison 0.0%). The code backs the claim that the same CTR means different things in different contexts.


In [5]:
# Same CTR, different meaning: median earlier-month CTR by tier x content type (cells with >= 30 pages)
ctx = lane.groupby(["position_tier", "content_type"])["ctr_prev"].agg(["size", "median"])
print("Median CTR (%) by position tier and content type:")
print(ctx[ctx["size"] >= 30].round(3).to_string())

# Volume noise: how often does a page's 'low CTR' status flip between the two months?
low_prev = lane["ctr_prev"] < 0.5 * lane["exp_prev"]
for name, mask in [("100-499 impressions", lane["impressions_prev_30d"] < 500), (">=500 impressions", lane["impressions_prev_30d"] >= 500)]:
    still = lane.loc[mask & low_prev, "under_captures_next"].mean()
    print(f"\n{name}: {int((mask & low_prev).sum()):,} pages under-captured in the earlier month; "
          f"{still:.0%} still did in the next month")


Median CTR (%) by position tier and content type:
                                  size  median
position_tier content_type                    
page_1        comparison article    43   0.000
              feedly article        81   0.299
              keyword article     6451   0.235
striking      feedly article        30   0.000
              keyword article     4128   0.139
top_3         keyword article      350   0.251

100-499 impressions: 1,654 pages under-captured in the earlier month; 65% still did in the next month

>=500 impressions: 2,181 pages under-captured in the earlier month; 57% still did in the next month


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.